# Open-Set Evaluation with HAPT Postural Transitions (TODO.md §10)

This notebook implements open-set detection using HAPT dataset transition classes as OOD samples.

**Setup:**
- **In-distribution**: Locomotion activities (WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS) from UCI HAR target subjects
- **Out-of-distribution**: Postural transitions (STAND_TO_SIT, SIT_TO_STAND, etc.) from HAPT same target subjects
- **Model**: 3-class locomotion classifier from §4 (reconstructed deterministically, seed=42)
- **Analysis**: AUROC of epistemic/aleatoric/total as OOD detectors, OOD/in-dist ratios

**NO new training** — reuses existing model architecture and training procedure.

**Multi-seed protocol** (5 seeds) applied to summarize AUROC statistics (same as §8).

## Setup

In [19]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
sys.path.insert(0, str(root_dir))

import numpy as np
import torch
from sklearn.metrics import roc_auc_score

from src.loader import load_har, LOCOMOTION_IDS
from src.hapt_loader import load_hapt, TRANSITION_IDS_HAPT, validate_hapt
from src.subject_split import split_and_scale
from src.bayesian import FeatureClassifier, LastLayerLaplace
from src.har_train import (
    subject_train_val_split,
    check_class_balance,
    train_source_model,
)

print("✓ Imports successful")

✓ Imports successful


## 1. Dataset Verification: HAPT Format and Subject IDs

In [20]:
# Load HAPT dataset
dataset_hapt = load_hapt(str(root_dir / "data"), merge_train_test=True)

print("HAPT Dataset Loaded:")
print(f"  Total samples: {dataset_hapt.X.shape[0]}")
print(f"  Features: {dataset_hapt.X.shape[1]}")
print(f"  Unique subjects: {len(np.unique(dataset_hapt.subject_id))}")
print(f"  Subject ID range: {dataset_hapt.subject_id.min()}-{dataset_hapt.subject_id.max()}")

HAPT Dataset Loaded:
  Total samples: 10929
  Features: 561
  Unique subjects: 30
  Subject ID range: 1-30


In [21]:
# Validate HAPT
hapt_validation = validate_hapt(dataset_hapt)

print("\nHAPT Validation Report:")
print("=" * 60)
for key, value in hapt_validation.items():
    if key == 'class_counts':
        print(f"{key}:")
        for act, count in value.items():
            print(f"    {act:25s}: {count:5d}")
    else:
        print(f"  {key:30s}: {value}")

# Verify subject IDs match UCI HAR (1-30)
hapt_subjects = sorted(np.unique(dataset_hapt.subject_id))
expected_subjects = list(range(1, 31))

if hapt_subjects == expected_subjects:
    print("\n✓ VERIFICATION PASSED: HAPT uses same 30 subjects as UCI HAR")
else:
    print(f"\n✗ VERIFICATION FAILED: Subject mismatch")
    print(f"  Expected: {expected_subjects}")
    print(f"  Found: {hapt_subjects}")
    raise RuntimeError("Subject ID mismatch between HAPT and UCI HAR")


HAPT Validation Report:
  n_rows                        : 10929
  n_features                    : 561
  n_nan                         : 0
  n_inf                         : 0
  min_val                       : -1.0
  max_val                       : 1.0
  n_outside_[-1,1]              : 0
  n_subjects                    : 30
  subject_id_range              : (np.int64(1), np.int64(30))
  activity_ids_present          : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
class_counts:
    WALKING                  :  1722
    WALKING_UPSTAIRS         :  1544
    WALKING_DOWNSTAIRS       :  1407
    SITTING                  :  1801
    STANDING                 :  1979
    LAYING                   :  1958
    STAND_TO_SIT             :    70
    SIT_TO_STAND             :    33
    SIT_TO_LIE               :   107
    LIE_TO_SIT               :    85
    STAND_TO_LIE             :   139
    LIE_TO_STAND             :    84

✓ VERIFICATION PASSED: HAPT uses same 30 subjects as UCI HAR


## 2. Multi-Seed Open-Set Evaluation

Train 5 models with different initialization seeds (same subject split seed=42) and compute AUROC statistics.

This follows the multi-seed protocol established in §8 (har_calibration.ipynb).

In [22]:
# Training seeds (same as §8)
TRAINING_SEEDS = [42, 123, 456, 789, 1011]
M_FIXED = 5000  # Same as §5
FINAL_SEED = 456  # For predictive computation

# Load UCI HAR (for in-distribution data)
dataset_har = load_har(str(root_dir / "data"), merge_train_test=True)
dataset_har_loco = dataset_har.filter_activities(LOCOMOTION_IDS)

# SAME split as §4 (seed=42 for subject partition)
split = split_and_scale(dataset_har_loco, n_source=20, seed=42)

print(f"Source subjects (seed=42): {split.source_subjects}")
print(f"Target subjects (seed=42): {split.target_subjects}")
print(f"\nTraining {len(TRAINING_SEEDS)} models with different initialization seeds...")

Source subjects (seed=42): [ 1  2  3  4  5  6  9 10 12 13 14 16 17 18 23 24 25 26 28 29]
Target subjects (seed=42): [ 7  8 11 15 19 20 21 22 27 30]

Training 5 models with different initialization seeds...


In [23]:
# Storage for multi-seed results
all_seed_results = []

label_map = {1: 0, 2: 1, 3: 2}

for seed_idx, train_seed in enumerate(TRAINING_SEEDS):
    print(f"\n{'='*60}")
    print(f"SEED {seed_idx+1}/{len(TRAINING_SEEDS)}: train_seed={train_seed}")
    print(f"{'='*60}")
    
    # Set seed for this run
    torch.manual_seed(train_seed)
    np.random.seed(train_seed)
    
    # SAME train/val split within source (seed=42, deterministic across seeds)
    train_subjects, val_subjects = subject_train_val_split(
        split.source_subjects, val_fraction=0.2, seed=42
    )
    
    train_mask = np.isin(split.subject_id_source, train_subjects)
    val_mask = np.isin(split.subject_id_source, val_subjects)
    
    X_train = torch.tensor(split.X_source[train_mask], dtype=torch.float32)
    y_train_raw = split.y_source[train_mask]
    y_train = torch.tensor([label_map[y] for y in y_train_raw], dtype=torch.long)
    
    X_val = torch.tensor(split.X_source[val_mask], dtype=torch.float32)
    y_val_raw = split.y_source[val_mask]
    y_val = torch.tensor([label_map[y] for y in y_val_raw], dtype=torch.long)
    
    # Check class balance
    class_weights = check_class_balance(y_train.numpy(), n_classes=3, split_name=f"train (seed={train_seed})")
    
    # Initialize and train model
    model = FeatureClassifier(in_dim=561, hidden_dims=[128, 64], n_classes=3)
    
    history = train_source_model(
        model, X_train, y_train, X_val, y_val,
        weight_decay=0.01, lr=1e-3, epochs=200,
        class_weights=class_weights, patience=20
    )
    
    print(f"\nTrained model (seed={train_seed}): tau_prior={history['tau_prior']:.2f}")
    
    # Fit Laplace
    X_source_full = torch.tensor(split.X_source, dtype=torch.float32)
    y_source_full = torch.tensor([label_map[y] for y in split.y_source], dtype=torch.long)
    
    laplace = LastLayerLaplace.fit(model, X_source_full, y_source_full,
                                   tau_prior=history['tau_prior'])
    
    # Collect epistemic/aleatoric/total for in-dist and OOD
    in_dist_epistemic = []
    in_dist_aleatoric = []
    in_dist_total = []
    
    ood_epistemic = []
    ood_aleatoric = []
    ood_total = []
    
    n_ood_samples_per_subject = {}
    
    for sid in sorted(split.target_subjects):
        # In-distribution: locomotion activities from UCI HAR
        ds_har_target = dataset_har_loco.filter_subjects([sid])
        X_har = torch.tensor(split.scaler.transform(ds_har_target.X), dtype=torch.float32)
        
        pred_har = laplace.predictive(
            model, X_har, M=M_FIXED, temperature=1.0,
            generator=torch.Generator().manual_seed(FINAL_SEED)
        )
        
        in_dist_epistemic.extend(pred_har['epistemic'].numpy())
        in_dist_aleatoric.extend(pred_har['aleatoric'].numpy())
        in_dist_total.extend(pred_har['total_entropy'].numpy())
        
        # OOD: transition activities from HAPT
        ds_hapt_target = dataset_hapt.filter_subjects([sid]).filter_activities(TRANSITION_IDS_HAPT)
        
        n_ood_samples_per_subject[int(sid)] = len(ds_hapt_target.y)
        
        if len(ds_hapt_target.y) < 10:
            print(f"  ⚠ WARNING: Subject {sid} has only {len(ds_hapt_target.y)} transition samples (<10)")
        
        X_hapt = torch.tensor(split.scaler.transform(ds_hapt_target.X), dtype=torch.float32)
        
        pred_hapt = laplace.predictive(
            model, X_hapt, M=M_FIXED, temperature=1.0,
            generator=torch.Generator().manual_seed(FINAL_SEED)
        )
        
        ood_epistemic.extend(pred_hapt['epistemic'].numpy())
        ood_aleatoric.extend(pred_hapt['aleatoric'].numpy())
        ood_total.extend(pred_hapt['total_entropy'].numpy())
    
    # Convert to arrays
    in_dist_epistemic = np.array(in_dist_epistemic)
    in_dist_aleatoric = np.array(in_dist_aleatoric)
    in_dist_total = np.array(in_dist_total)
    ood_epistemic = np.array(ood_epistemic)
    ood_aleatoric = np.array(ood_aleatoric)
    ood_total = np.array(ood_total)
    
    # Compute AUROC (1 = in-dist, 0 = OOD)
    y_true = np.concatenate([np.ones(len(in_dist_epistemic)), np.zeros(len(ood_epistemic))])
    
    # Higher uncertainty should indicate OOD, so negate for AUROC
    scores_epistemic = -np.concatenate([in_dist_epistemic, ood_epistemic])
    scores_aleatoric = -np.concatenate([in_dist_aleatoric, ood_aleatoric])
    scores_total = -np.concatenate([in_dist_total, ood_total])
    
    auroc_epistemic = roc_auc_score(y_true, scores_epistemic)
    auroc_aleatoric = roc_auc_score(y_true, scores_aleatoric)
    auroc_total = roc_auc_score(y_true, scores_total)
    
    # Ratios OOD/in-dist
    ratio_epistemic = ood_epistemic.mean() / in_dist_epistemic.mean()
    ratio_aleatoric = ood_aleatoric.mean() / in_dist_aleatoric.mean()
    ratio_total = ood_total.mean() / in_dist_total.mean()
    
    print(f"\nResults (seed={train_seed}), reference = TARGET known activities:")
    print(f"  In-dist samples: {len(in_dist_epistemic)}")
    print(f"  OOD samples: {len(ood_epistemic)}")
    print(f"  AUROC epistemic: {auroc_epistemic:.4f}")
    print(f"  AUROC aleatoric: {auroc_aleatoric:.4f}")
    print(f"  AUROC total: {auroc_total:.4f}")
    print(f"  OOD/in-dist ratio epistemic: {ratio_epistemic:.3f}")
    print(f"  OOD/in-dist ratio aleatoric: {ratio_aleatoric:.3f}")
    print(f"  OOD/in-dist ratio total: {ratio_total:.3f}")
    

    # SECOND COMPARISON: SOURCE validation set as in-distribution reference
    # (same model/Laplace already fitted, same OOD group, different in-dist reference)
    pred_source_val = laplace.predictive(
        model, X_val, M=M_FIXED, temperature=1.0,
        generator=torch.Generator().manual_seed(FINAL_SEED)
    )

    source_val_epistemic = pred_source_val['epistemic'].numpy()
    source_val_aleatoric = pred_source_val['aleatoric'].numpy()
    source_val_total = pred_source_val['total_entropy'].numpy()

    # Compute AUROC with source val as in-dist, HAPT transitions as OOD
    y_true_srcval = np.concatenate([np.ones(len(source_val_epistemic)), np.zeros(len(ood_epistemic))])

    scores_epistemic_srcval = -np.concatenate([source_val_epistemic, ood_epistemic])
    scores_aleatoric_srcval = -np.concatenate([source_val_aleatoric, ood_aleatoric])
    scores_total_srcval = -np.concatenate([source_val_total, ood_total])

    auroc_epistemic_srcval = roc_auc_score(y_true_srcval, scores_epistemic_srcval)
    auroc_aleatoric_srcval = roc_auc_score(y_true_srcval, scores_aleatoric_srcval)
    auroc_total_srcval = roc_auc_score(y_true_srcval, scores_total_srcval)

    # Ratios OOD/source-val
    ratio_epistemic_srcval = ood_epistemic.mean() / source_val_epistemic.mean()
    ratio_aleatoric_srcval = ood_aleatoric.mean() / source_val_aleatoric.mean()
    ratio_total_srcval = ood_total.mean() / source_val_total.mean()

    print(f"\nResults (seed={train_seed}), reference = SOURCE validation:")
    print(f"  In-dist samples: {len(source_val_epistemic)}")
    print(f"  OOD samples: {len(ood_epistemic)}")
    print(f"  AUROC epistemic: {auroc_epistemic_srcval:.4f}")
    print(f"  AUROC aleatoric: {auroc_aleatoric_srcval:.4f}")
    print(f"  AUROC total: {auroc_total_srcval:.4f}")
    print(f"  OOD/in-dist ratio epistemic: {ratio_epistemic_srcval:.3f}")
    print(f"  OOD/in-dist ratio aleatoric: {ratio_aleatoric_srcval:.3f}")
    print(f"  OOD/in-dist ratio total: {ratio_total_srcval:.3f}")

    all_seed_results.append({
        'train_seed': train_seed,
        'auroc_epistemic': auroc_epistemic,
        'auroc_aleatoric': auroc_aleatoric,
        'auroc_total': auroc_total,
        'ratio_epistemic': ratio_epistemic,
        'ratio_aleatoric': ratio_aleatoric,
        'ratio_total': ratio_total,
        'n_in_dist': len(in_dist_epistemic),
        'n_ood': len(ood_epistemic),
        'n_ood_per_subject': n_ood_samples_per_subject,
        # Source val reference
        'auroc_epistemic_srcval': auroc_epistemic_srcval,
        'auroc_aleatoric_srcval': auroc_aleatoric_srcval,
        'auroc_total_srcval': auroc_total_srcval,
        'ratio_epistemic_srcval': ratio_epistemic_srcval,
        'ratio_aleatoric_srcval': ratio_aleatoric_srcval,
        'ratio_total_srcval': ratio_total_srcval,
        'n_source_val': len(source_val_epistemic),
    })

print(f"\n{'='*60}")
print(f"✓ Multi-seed evaluation complete ({len(TRAINING_SEEDS)} seeds)")
print(f"{'='*60}")


SEED 1/5: train_seed=42

Class distribution (train (seed=42)):
  Counts: [910 839 761]
  Frequencies: [0.3625498  0.33426295 0.30318725]
  Imbalance ratio (max/min): 1.20
  Classes reasonably balanced, no weighting needed

Training configuration:
  Optimizer: AdamW
  Learning rate: 0.001
  Weight decay: 0.01
  tau_prior (weight_decay * N_train): 25.10
  Max epochs: 200
  Early stopping patience: 20
  Train samples: 2510, Val samples: 687
Epoch   1/200: train_loss=1.0916 train_acc=0.284 train_recall=0.277 | val_loss=0.9861 val_acc=0.783 val_recall=0.773 | patience=0/20
Epoch  10/200: train_loss=0.4553 train_acc=0.880 train_recall=0.877 | val_loss=0.3723 val_acc=0.948 val_recall=0.942 | patience=0/20
Epoch  20/200: train_loss=0.1321 train_acc=0.973 train_recall=0.972 | val_loss=0.1741 val_acc=0.946 val_recall=0.942 | patience=0/20
Epoch  30/200: train_loss=0.0358 train_acc=0.995 train_recall=0.995 | val_loss=0.0863 val_acc=0.971 val_recall=0.969 | patience=2/20
Epoch  40/200: train_loss

## 3. Multi-Seed Summary Statistics

In [24]:
# Aggregate across seeds
auroc_epistemic_all = [r['auroc_epistemic'] for r in all_seed_results]
auroc_aleatoric_all = [r['auroc_aleatoric'] for r in all_seed_results]
auroc_total_all = [r['auroc_total'] for r in all_seed_results]

ratio_epistemic_all = [r['ratio_epistemic'] for r in all_seed_results]
ratio_aleatoric_all = [r['ratio_aleatoric'] for r in all_seed_results]
ratio_total_all = [r['ratio_total'] for r in all_seed_results]

print("\n" + "=" * 60)
print("MULTI-SEED SUMMARY (N=5 seeds)")
print("=" * 60)

print(f"\nAUROC (OOD Detection):")
print(f"  Epistemic: {np.mean(auroc_epistemic_all):.4f} ± {np.std(auroc_epistemic_all):.4f}")
print(f"  Aleatoric: {np.mean(auroc_aleatoric_all):.4f} ± {np.std(auroc_aleatoric_all):.4f}")
print(f"  Total:     {np.mean(auroc_total_all):.4f} ± {np.std(auroc_total_all):.4f}")

print(f"\nOOD/In-Dist Ratios:")
print(f"  Epistemic: {np.mean(ratio_epistemic_all):.3f} ± {np.std(ratio_epistemic_all):.3f}")
print(f"  Aleatoric: {np.mean(ratio_aleatoric_all):.3f} ± {np.std(ratio_aleatoric_all):.3f}")
print(f"  Total:     {np.mean(ratio_total_all):.3f} ± {np.std(ratio_total_all):.3f}")

print(f"\nSample counts (consistent across seeds):")
print(f"  In-distribution: {all_seed_results[0]['n_in_dist']}")
print(f"  OOD (transitions): {all_seed_results[0]['n_ood']}")

print(f"\nOOD samples per target subject:")

# Aggregate for source val reference
auroc_epistemic_srcval_all = [r['auroc_epistemic_srcval'] for r in all_seed_results]
auroc_aleatoric_srcval_all = [r['auroc_aleatoric_srcval'] for r in all_seed_results]
auroc_total_srcval_all = [r['auroc_total_srcval'] for r in all_seed_results]

ratio_epistemic_srcval_all = [r['ratio_epistemic_srcval'] for r in all_seed_results]
ratio_aleatoric_srcval_all = [r['ratio_aleatoric_srcval'] for r in all_seed_results]
ratio_total_srcval_all = [r['ratio_total_srcval'] for r in all_seed_results]

print("\n" + "=" * 60)
print("SECOND COMPARISON: SOURCE validation as reference")
print("=" * 60)

print(f"\nAUROC (OOD Detection):")
print(f"  Epistemic: {np.mean(auroc_epistemic_srcval_all):.4f} ± {np.std(auroc_epistemic_srcval_all):.4f}")
print(f"  Aleatoric: {np.mean(auroc_aleatoric_srcval_all):.4f} ± {np.std(auroc_aleatoric_srcval_all):.4f}")
print(f"  Total:     {np.mean(auroc_total_srcval_all):.4f} ± {np.std(auroc_total_srcval_all):.4f}")

print(f"\nOOD/In-Dist Ratios:")
print(f"  Epistemic: {np.mean(ratio_epistemic_srcval_all):.3f} ± {np.std(ratio_epistemic_srcval_all):.3f}")
print(f"  Aleatoric: {np.mean(ratio_aleatoric_srcval_all):.3f} ± {np.std(ratio_aleatoric_srcval_all):.3f}")
print(f"  Total:     {np.mean(ratio_total_srcval_all):.3f} ± {np.std(ratio_total_srcval_all):.3f}")

print(f"\nSample counts:")
print(f"  In-distribution (source val): {all_seed_results[0]['n_source_val']}")
print(f"  OOD (transitions): {all_seed_results[0]['n_ood']}")

# Interpretation (factual, not speculative)
print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

target_auroc = np.mean(auroc_epistemic_all)
srcval_auroc = np.mean(auroc_epistemic_srcval_all)

print(f"\nTarget reference AUROC (epistemic): {target_auroc:.4f}")
print(f"Source val reference AUROC (epistemic): {srcval_auroc:.4f}")

if srcval_auroc < 0.5 and target_auroc < 0.5:
    print("\n→ Both references show inverted AUROC (<0.5).")
    print("→ This supports Hypothesis (1): Laplace interpolation limit.")
    print("→ HAPT transitions fall within feature space convex hull.")
elif srcval_auroc > 0.5 and target_auroc < 0.5:
    print("\n→ Source val reference AUROC >0.5, target reference <0.5.")
    print("→ This supports Hypothesis (2): Target reference was confounded.")
    print("→ Subject shift in target masked the class novelty signal.")
else:
    print(f"\n→ Unexpected pattern: srcval={srcval_auroc:.3f}, target={target_auroc:.3f}")

for sid in sorted(all_seed_results[0]['n_ood_per_subject'].keys()):
    n = all_seed_results[0]['n_ood_per_subject'][sid]
    flag = " ⚠ (<10)" if n < 10 else ""
    print(f"  Subject {sid:2d}: {n:3d} samples{flag}")


MULTI-SEED SUMMARY (N=5 seeds)

AUROC (OOD Detection):
  Epistemic: 0.0938 ± 0.0246
  Aleatoric: 0.0888 ± 0.0196
  Total:     0.0899 ± 0.0215

OOD/In-Dist Ratios:
  Epistemic: 0.325 ± 0.061
  Aleatoric: 0.170 ± 0.010
  Total:     0.205 ± 0.007

Sample counts (consistent across seeds):
  In-distribution: 1475
  OOD (transitions): 160

OOD samples per target subject:

SECOND COMPARISON: SOURCE validation as reference

AUROC (OOD Detection):
  Epistemic: 0.0799 ± 0.0171
  Aleatoric: 0.0716 ± 0.0128
  Total:     0.0739 ± 0.0141

OOD/In-Dist Ratios:
  Epistemic: 1.062 ± 0.142
  Aleatoric: 0.192 ± 0.010
  Total:     0.274 ± 0.011

Sample counts:
  In-distribution (source val): 687
  OOD (transitions): 160

INTERPRETATION

Target reference AUROC (epistemic): 0.0938
Source val reference AUROC (epistemic): 0.0799

→ Both references show inverted AUROC (<0.5).
→ This supports Hypothesis (1): Laplace interpolation limit.
→ HAPT transitions fall within feature space convex hull.
  Subject  7:  12

## 4. Save Open-Set Artifact

Save BALD artifact with in-dist and OOD samples (using first seed for consistency with other artifacts).

In [25]:
# Use first seed to generate artifact (seed=42, for consistency)
print("Regenerating artifact with seed=42 for storage...\n")

torch.manual_seed(42)
np.random.seed(42)

# Reconstruct model (seed=42)
train_subjects, val_subjects = subject_train_val_split(
    split.source_subjects, val_fraction=0.2, seed=42
)

train_mask = np.isin(split.subject_id_source, train_subjects)
val_mask = np.isin(split.subject_id_source, val_subjects)

X_train = torch.tensor(split.X_source[train_mask], dtype=torch.float32)
y_train_raw = split.y_source[train_mask]
y_train = torch.tensor([label_map[y] for y in y_train_raw], dtype=torch.long)

X_val = torch.tensor(split.X_source[val_mask], dtype=torch.float32)
y_val_raw = split.y_source[val_mask]
y_val = torch.tensor([label_map[y] for y in y_val_raw], dtype=torch.long)

class_weights = check_class_balance(y_train.numpy(), n_classes=3, split_name="final artifact")

model_final = FeatureClassifier(in_dim=561, hidden_dims=[128, 64], n_classes=3)

history_final = train_source_model(
    model_final, X_train, y_train, X_val, y_val,
    weight_decay=0.01, lr=1e-3, epochs=200,
    class_weights=class_weights, patience=20
)

X_source_full = torch.tensor(split.X_source, dtype=torch.float32)
y_source_full = torch.tensor([label_map[y] for y in split.y_source], dtype=torch.long)

laplace_final = LastLayerLaplace.fit(model_final, X_source_full, y_source_full,
                                     tau_prior=history_final['tau_prior'])

print(f"\n✓ Model reconstructed (seed=42): tau_prior={history_final['tau_prior']:.2f}")

Regenerating artifact with seed=42 for storage...


Class distribution (final artifact):
  Counts: [910 839 761]
  Frequencies: [0.3625498  0.33426295 0.30318725]
  Imbalance ratio (max/min): 1.20
  Classes reasonably balanced, no weighting needed

Training configuration:
  Optimizer: AdamW
  Learning rate: 0.001
  Weight decay: 0.01
  tau_prior (weight_decay * N_train): 25.10
  Max epochs: 200
  Early stopping patience: 20
  Train samples: 2510, Val samples: 687
Epoch   1/200: train_loss=1.0916 train_acc=0.284 train_recall=0.277 | val_loss=0.9861 val_acc=0.783 val_recall=0.773 | patience=0/20
Epoch  10/200: train_loss=0.4553 train_acc=0.880 train_recall=0.877 | val_loss=0.3723 val_acc=0.948 val_recall=0.942 | patience=0/20
Epoch  20/200: train_loss=0.1321 train_acc=0.973 train_recall=0.972 | val_loss=0.1741 val_acc=0.946 val_recall=0.942 | patience=0/20
Epoch  30/200: train_loss=0.0358 train_acc=0.995 train_recall=0.995 | val_loss=0.0863 val_acc=0.971 val_recall=0.969 | patience=2/20


In [26]:
# Generate artifact
artifact_openset = {}

for sid in sorted(split.target_subjects):
    # In-distribution
    ds_har_target = dataset_har_loco.filter_subjects([sid])
    X_har = torch.tensor(split.scaler.transform(ds_har_target.X), dtype=torch.float32)
    y_har = np.array([label_map[y] for y in ds_har_target.y])
    
    pred_har = laplace_final.predictive(
        model_final, X_har, M=M_FIXED, temperature=1.0,
        generator=torch.Generator().manual_seed(FINAL_SEED)
    )
    
    artifact_openset[f'in_dist_{sid}_X'] = X_har.numpy()
    artifact_openset[f'in_dist_{sid}_y'] = y_har
    artifact_openset[f'in_dist_{sid}_mean_probs'] = pred_har['mean_probs'].numpy()
    artifact_openset[f'in_dist_{sid}_total_entropy'] = pred_har['total_entropy'].numpy()
    artifact_openset[f'in_dist_{sid}_aleatoric'] = pred_har['aleatoric'].numpy()
    artifact_openset[f'in_dist_{sid}_epistemic'] = pred_har['epistemic'].numpy()
    
    # OOD (transitions)
    ds_hapt_target = dataset_hapt.filter_subjects([sid]).filter_activities(TRANSITION_IDS_HAPT)
    X_hapt = torch.tensor(split.scaler.transform(ds_hapt_target.X), dtype=torch.float32)
    y_hapt = ds_hapt_target.y  # Keep original HAPT labels (7-12)
    
    pred_hapt = laplace_final.predictive(
        model_final, X_hapt, M=M_FIXED, temperature=1.0,
        generator=torch.Generator().manual_seed(FINAL_SEED)
    )
    
    artifact_openset[f'ood_{sid}_X'] = X_hapt.numpy()
    artifact_openset[f'ood_{sid}_y'] = y_hapt
    artifact_openset[f'ood_{sid}_mean_probs'] = pred_hapt['mean_probs'].numpy()
    artifact_openset[f'ood_{sid}_total_entropy'] = pred_hapt['total_entropy'].numpy()
    artifact_openset[f'ood_{sid}_aleatoric'] = pred_hapt['aleatoric'].numpy()
    artifact_openset[f'ood_{sid}_epistemic'] = pred_hapt['epistemic'].numpy()


# Source validation set predictions (for second comparison)
pred_source_val_final = laplace_final.predictive(
    model_final, X_val, M=M_FIXED, temperature=1.0,
    generator=torch.Generator().manual_seed(FINAL_SEED)
)

artifact_openset['source_val_X'] = X_val.numpy()
artifact_openset['source_val_y'] = y_val.numpy()
artifact_openset['source_val_mean_probs'] = pred_source_val_final['mean_probs'].numpy()
artifact_openset['source_val_total_entropy'] = pred_source_val_final['total_entropy'].numpy()
artifact_openset['source_val_aleatoric'] = pred_source_val_final['aleatoric'].numpy()
artifact_openset['source_val_epistemic'] = pred_source_val_final['epistemic'].numpy()

# Metadata
artifact_openset['target_subject_ids'] = sorted(split.target_subjects)
artifact_openset['M_FIXED'] = M_FIXED
artifact_openset['tau_prior'] = history_final['tau_prior']
artifact_openset['seed'] = 42
artifact_openset['final_seed'] = FINAL_SEED

# Multi-seed statistics
artifact_openset['auroc_epistemic_mean'] = np.mean(auroc_epistemic_all)
artifact_openset['auroc_epistemic_std'] = np.std(auroc_epistemic_all)
artifact_openset['auroc_aleatoric_mean'] = np.mean(auroc_aleatoric_all)
artifact_openset['auroc_aleatoric_std'] = np.std(auroc_aleatoric_all)
artifact_openset['auroc_total_mean'] = np.mean(auroc_total_all)
artifact_openset['auroc_total_std'] = np.std(auroc_total_all)

# Source val reference statistics
artifact_openset['auroc_epistemic_srcval_mean'] = np.mean(auroc_epistemic_srcval_all)
artifact_openset['auroc_epistemic_srcval_std'] = np.std(auroc_epistemic_srcval_all)
artifact_openset['auroc_aleatoric_srcval_mean'] = np.mean(auroc_aleatoric_srcval_all)
artifact_openset['auroc_aleatoric_srcval_std'] = np.std(auroc_aleatoric_srcval_all)
artifact_openset['auroc_total_srcval_mean'] = np.mean(auroc_total_srcval_all)
artifact_openset['auroc_total_srcval_std'] = np.std(auroc_total_srcval_all)

# Save
output_path = root_dir / "data" / "har_bald_artifact_openset.npz"
np.savez_compressed(output_path, **artifact_openset)

print(f"\n✓ Saved open-set artifact to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")


✓ Saved open-set artifact to: /Users/riccardo/Desktop/PML/BayesianExoAdaptation/data/har_bald_artifact_openset.npz
  File size: 4553.3 KB


## Summary

Open-set evaluation complete:
- ✓ HAPT dataset verified (same 30 subjects, 561 features)
- ✓ Multi-seed protocol (5 seeds) applied
- ✓ AUROC and OOD/in-dist ratios computed
- ✓ Artifact saved to `data/har_bald_artifact_openset.npz`

**NO interpretation here** — results reported above for review.